In [1]:
import torch
from molhiv.utils import calculate_label_imbalance, prec, acc, rec, roc_auc, download_graph_prop_pred_dataset
from molhiv.ginenn import GINENN
from molhiv.training import Metric, train_val
import torch.nn as nn
import os
import numpy as np
from molhiv.training import predict
import yaml

with open("../configs/molhiv.yaml") as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
dataset = download_graph_prop_pred_dataset()

split_idx = dataset.get_idx_split()

from torch_geometric.loader import DataLoader
size = cfg["data"]["datasize"]

train_dataset = dataset[split_idx["train"]][:size]
val_dataset = dataset[split_idx["valid"]][:size]
test_dataset = dataset[split_idx["test"]]

sample_labels = [dataset[i].y.squeeze() for i in range(len(train_dataset))]
class_weights, sample_weights = calculate_label_imbalance(sample_labels)
class_weights = class_weights.to(device)

from torch.utils.data import WeightedRandomSampler
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=cfg["data"]["batch_size"], sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=cfg["data"]["batch_size"])
test_loader = DataLoader(test_dataset, batch_size=cfg["data"]["batch_size"])

model = GINENN(**cfg["model"])
optimizer = torch.optim.Adam(model.parameters(), lr=cfg["optimizer"]["lr"], weight_decay=cfg["optimizer"]["weight_decay"])

metrics = [
    Metric("train_prec", fn=prec, split="train"),
    Metric("val_prec", fn=prec, split="val"),

    Metric("train_acc", fn=acc, split="train"),
    Metric("val_acc", fn=acc, split="val"),

    Metric("train_rec", fn=rec, split="train"),
    Metric("val_rec", fn=rec, split="val"),

    Metric("train_roc_auc", fn=roc_auc, split="train"),
    Metric("val_roc_auc", fn=roc_auc, split="val")
]
neg_counts, pos_counts = np.unique(sample_labels, return_counts=True)[1]
pos_weight = torch.tensor([neg_counts / pos_counts], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, **cfg["scheduler"]
)

import mlflow
# mlflow.set_tracking_uri(f"file://{os.path.expanduser('~')}/projects/molhiv/mlruns")
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Molhiv-GCN-HIV-binding")
with mlflow.start_run(run_name="Overfitting-testing"):
    mlflow.log_params(cfg["data"])
    mlflow.log_params(cfg["model"])
    mlflow.log_params(cfg["optimizer"])
    mlflow.log_params(cfg["scheduler"])
    mlflow.log_params(cfg["training"])

    for epoch in range(cfg["training"]["epochs"]):
        results = train_val(model, train_loader, val_loader, optimizer, criterion, metrics, cfg["training"]["max_grad_norm"], device)
        mlflow.log_metrics(results, step=epoch)
    
        scheduler.step(results["val_loss"])
        mlflow.log_metric("lr_per_epoch", scheduler.optimizer.param_groups[0]["lr"], step=epoch)

    prob, y_true = predict(model, test_loader, device)

    test_roc_auc = roc_auc(prob, y_true)
    mlflow.log_metric("test_roc_auc", test_roc_auc)


/Users/tungvuduc/opt/anaconda3/envs/dlbio_arm64/lib/python3.12/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/tungvuduc/opt/anaconda3/envs/dlbio_arm64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cpu
🏃 View run Overfitting-testing at: http://127.0.0.1:5000/#/experiments/6/runs/795db075769348e1b7c1286b6bf023fe
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6
